In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from delta.tables import *
from pyspark.sql.window import*

### Scenario:
You're ingesting `transactions_mixed_types.json` from an upstream source with two real-world data quality problems at once: **inconsistent field types** (`amount` shows up as a number in some records and as a string in others), and a **malformed record** (one line has broken/invalid JSON syntax entirely — e.g., a missing closing brace). A naive read will either crash or silently produce nulls; you need to explicitly capture and quarantine the malformed line while still successfully processing everything else, and normalize the type inconsistency.

**Problem:**

- Define an explicit schema where `amount` is read as **`StringType`** (this is the safe move when a field's type is inconsistent across records — capture it as a string first, cast it properly afterward, rather than letting the reader guess and silently fail on one of the two representations).
- Include `_corrupt_record` (`StringType`) as an extra field in your schema, and set the read option `mode` to `PERMISSIVE` with `columnNameOfCorruptRecord` set to `_corrupt_record` — this tells Spark to route unparseable lines into that column instead of dropping or erroring on them.
- Split the result into two sets:
  - **Corrupt rows** — where `_corrupt_record` is not null. Write these to a `transactions_quarantine` Delta table.
  - **Valid rows** — where `_corrupt_record` is null.
- For valid rows, cast `amount` (currently a string, regardless of whether it originally arrived as a number or a string in the JSON) to `double`.
- Compute `total_amount` — the sum of `amount` across all valid rows.

**Expected Output — quarantine count**

| corrupt_count |
| :--- |
| 1 |

**Expected Output — total amount (valid rows only)**

| total_amount |
| :--- |
| 440.5 |

In [0]:
json_schema = StructType(
    [
        StructField("txn_id", StringType()),
        StructField("customer_id", StringType()),
        StructField("amount", StringType()),
        StructField("txn_date", DateType()),
        StructField("_corrupt_record", StringType())
    ]
)

df = spark.read.format("json").schema(json_schema) \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .load("/Workspace/Users/jeevan.busi8008@gmail.com/spark-practice/data/transactions_mixed_types.json")

valid_txn_df = df.filter(col("_corrupt_record").isNull())
invalid_txn_df = df.filter(col("_corrupt_record").isNotNull())

## loading into delta table as counting the corrupted rows is not possible in serverless compute.
invalid_txn_df.write.format("delta").mode("overwrite").saveAsTable("pyspark_practice.default.transactions_quarantine")
corrupt_count = spark.read.table("pyspark_practice.default.transactions_quarantine").filter(col("_corrupt_record").isNotNull()).count()

print(f"corrupt_count: {corrupt_count}")


valid_txn_df = valid_txn_df.withColumn("amount", col("amount").cast(DecimalType(10,2)))
valid_txn_df.agg(sum(col("amount")).alias("total_amount")).show()